In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_dublin_core.xml", "data/metadata/climate_dataset_dublin_core.xml"),
    ("data/metadata/climate_dataset_datacite.xml", "data/metadata/climate_dataset_datacite.xml"),
    ("data/metadata/climate_dataset_schemaorg.jsonld", "data/metadata/climate_dataset_schemaorg.jsonld"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_dublin_core.xml
Downloaded: data/metadata/climate_dataset_datacite.xml
Downloaded: data/metadata/climate_dataset_schemaorg.jsonld
Bootstrap complete.


# Day 2: Compare Dublin Core, DataCite, and schema.org

This notebook loads the three local metadata examples for the teaching dataset and maps them into a common comparison table.

We focus on shared fields such as `title`, `creator`, `identifier`, `description`, `publisher`, `date/year`, and `format`.

### Colab tip
Upload all three metadata files to `/content/` if you are not running from the repository directory structure.


In [2]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd

# ---- Notebook-local parsers to avoid src import issues in Colab ----
def _local_name(tag: str) -> str:
    return tag.split('}', 1)[1] if '}' in tag else tag

def parse_dc(xml_path):
    root = ET.parse(xml_path).getroot()
    def first(name):
        for el in root.iter():
            if _local_name(el.tag) == name:
                txt = (el.text or '').strip()
                if txt:
                    return txt
        return None
    creators = []
    for el in root.iter():
        if _local_name(el.tag) == 'creator':
            txt = (el.text or '').strip()
            if txt:
                creators.append(txt)
    return {
        'standard': 'Dublin Core',
        'title': first('title'),
        'creator': creators,
        'identifier': first('identifier'),
        'description': first('description'),
        'publisher': first('publisher'),
        'date_or_year': first('date'),
        'format': first('format'),
    }

def parse_datacite(xml_path):
    root = ET.parse(xml_path).getroot()
    out = {'standard': 'DataCite', 'creator': []}
    for el in root.iter():
        name = _local_name(el.tag)
        txt = (el.text or '').strip()
        if name == 'identifier' and txt and 'identifier' not in out:
            out['identifier'] = txt
        elif name == 'creatorName' and txt:
            out['creator'].append(txt)
        elif name == 'title' and txt and 'title' not in out:
            out['title'] = txt
        elif name == 'description' and txt and 'description' not in out:
            out['description'] = txt
        elif name == 'publisher' and txt and 'publisher' not in out:
            out['publisher'] = txt
        elif name == 'publicationYear' and txt and 'date_or_year' not in out:
            out['date_or_year'] = txt
        elif name == 'resourceType' and 'resourceType' not in out:
            out['resourceType'] = el.attrib.get('resourceTypeGeneral') or txt
    out.setdefault('title', None)
    out.setdefault('identifier', None)
    out.setdefault('description', None)
    out.setdefault('publisher', None)
    out.setdefault('date_or_year', None)
    out.setdefault('format', None)
    return out

def parse_schema(json_path):
    obj = json.loads(Path(json_path).read_text(encoding='utf-8'))
    creators = []
    for c in obj.get('creator', []):
        if isinstance(c, dict) and c.get('name'):
            creators.append(c['name'])
    format_value = None
    dist = obj.get('distribution', [])
    if isinstance(dist, list) and dist and isinstance(dist[0], dict):
        format_value = dist[0].get('encodingFormat')
    return {
        'standard': 'schema.org',
        'title': obj.get('name'),
        'creator': creators,
        'identifier': obj.get('identifier'),
        'description': obj.get('description'),
        'publisher': obj.get('publisher', {}).get('name') if isinstance(obj.get('publisher'), dict) else obj.get('publisher'),
        'date_or_year': None,
        'format': format_value,
        'resourceType': obj.get('@type'),
    }

def compare_three_standards(dc, datacite, schema):
    rows = [dc, datacite, schema]
    for r in rows:
        if isinstance(r.get('creator'), list):
            r['creator'] = '; '.join(r['creator'])
    cols = ['standard', 'title', 'creator', 'identifier', 'description', 'publisher', 'date_or_year', 'format', 'resourceType']
    return pd.DataFrame(rows)[cols]

# Local repo paths with Colab fallback path hints
dc_path = Path('/content/data/metadata/climate_dataset_dublin_core.xml')
dcite_path = Path('/content/data/metadata/climate_dataset_datacite.xml')
schema_path = Path('/content/data/metadata/climate_dataset_schemaorg.jsonld')
if not dc_path.exists():
    dc_path = Path('/content/climate_dataset_dublin_core.xml')
if not dcite_path.exists():
    dcite_path = Path('/content/climate_dataset_datacite.xml')
if not schema_path.exists():
    schema_path = Path('/content/climate_dataset_schemaorg.jsonld')

dc_meta = parse_dc(dc_path)
datacite_meta = parse_datacite(dcite_path)
schema_meta = parse_schema(schema_path)

df_compare = compare_three_standards(dc_meta, datacite_meta, schema_meta)
df_compare

,standard,title,creator,identifier,description,publisher,date_or_year,format,resourceType
0,Dublin Core,2025 Global Climate Data,Global Climate Data Team,10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),2025-12-31,text/csv,NaN
1,DataCite,2025 Global Climate Data,Global Climate Data Team,10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),2025,None,Dataset
2,schema.org,2025 Global Climate Data,Global Climate Data Team,10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),None,text/csv,Dataset


## Similarities and differences (short interpretation)

- All three standards include a clear dataset `title`.
- All three standards include a persistent identifier (`identifier`).
- Dublin Core emphasizes simple descriptive fields; DataCite emphasizes DOI/citation-style fields; schema.org emphasizes web discovery fields.
- Differences may appear in field names (`publicationYear` vs `date`) and in where formats are represented.
